In [ ]:
import base64, hashlib, io, json, os, pathlib, subprocess, sys, zipfile
for key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[key] = '2'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
directory = pathlib.Path('/tmp/exp052-fixed-inputs')
directory.mkdir(exist_ok=False)
raw = base64.b64decode('...', validate=True)
assert hashlib.sha256(raw).hexdigest() == '2c86861bf051df0c033884f219e98bd518ebfaf9242f478424a649a9f995dc51'
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    for name in archive.namelist():
        assert pathlib.PurePosixPath(name).name == name
        (directory/name).write_bytes(archive.read(name))
cfg = json.loads((directory/'config.json').read_text())
inputs = pathlib.Path('/kaggle/input')
for name in ('train.csv','folds.csv','extra.csv'):
    matches = [p for p in inputs.rglob(name) if hashlib.sha256(p.read_bytes()).hexdigest() == cfg['input_hashes'][name]]
    assert len(matches) == 1, ('Frozen input mismatch',name,len(matches))
    (directory/name).write_bytes(matches[0].read_bytes())
for name, expected in cfg['input_hashes'].items():
    assert hashlib.sha256((directory/name).read_bytes()).hexdigest() == expected, name
output = pathlib.Path('/kaggle/working/results')
print('EXP052 fixed CPU run, starting the two four-model training arms', flush=True)
try:
    subprocess.run([sys.executable,'-u',str(directory/'train_volume.py'),'--payload',str(directory),'--output',str(output)],check=True,timeout=6660)
finally:
    if output.is_dir():
        with zipfile.ZipFile('/kaggle/working/exp052-results.zip','w',zipfile.ZIP_DEFLATED) as archive:
            for path in sorted(output.rglob('*')):
                if path.is_file():
                    archive.write(path,path.relative_to(output).as_posix())
